# 🇮🇹 Everyday Italian — Pimsleur-style audio lesson generator (B1)

**Lesson 6:** morning routine & personal care · daily activities & evenings · hobbies · politics & government (how Italy is governed, voting, laws, polite disagreement) · rules (*vietato, obbligatorio, raccolta differenziata*) · religion & spirituality · small talk with neighbours, party guests, other parents, the teacher, teammates

Grammar in context: reflexive verbs in present and past (*mi alzo, mi sono alzato*), causative *farsi* (*mi faccio tagliare i capelli*), *mi piace / mi piacciono*, *prima di* + inf., *dopo aver* + participle, passive (*è stata approvata*), *credo che / non credo che* + subjunctive, *si dice che*, *non si può*. Part 11 reviews sentences from Lessons 1–5.

How the audio works:
* **English cue → pause → Italian answer → pause.** The first pause is for you to *say* the sentence, the second for you to *repeat* after the native speaker.
* **Pause lengths depend on the Italian.** Each pause is computed from the measured duration of the Italian clip:
  * answer pause = `PAUSE_SCALE × clamp(THINK_MS + SAY_FACTOR × d_it)`
  * repeat pause = `PAUSE_SCALE × clamp(REPEAT_BASE_MS + REPEAT_FACTOR × d_it)`
* **Two Italian voices** (female/male, Microsoft neural voices via `edge-tts`) plus an English narrator; falls back to `gTTS` automatically if a clip fails.
* Back-chaining (`ingegneria → gegneria → ...`), repeated words (`x2`, `x3`, optional slow repeat), spaced review of earlier sentences, role-play, and the dialogue at start and end.

**Run:** Runtime → Run all. The MP3 and a time-stamped transcript are downloaded at the end (~3–5 min synthesis for ~420 clips; the cache makes reruns fast).


In [ ]:
!pip -q install edge-tts gTTS pydub
# ffmpeg is pre-installed on Colab (pydub needs it)

## 1 · Settings

In [ ]:
#@title Voices, speed and pause settings { display-mode: "form" }
LESSON_NAME   = "Lesson06_VitaSociale_B1"  #@param {type:"string"}
ENGINE        = "edge"   #@param ["edge", "gtts"]
VOICE_EN      = "en-US-AndrewNeural"  #@param ["en-US-AndrewNeural", "en-US-GuyNeural", "en-US-JennyNeural", "en-GB-RyanNeural"]
VOICE_IT_F    = "it-IT-ElsaNeural"    #@param ["it-IT-ElsaNeural", "it-IT-IsabellaNeural"]
VOICE_IT_M    = "it-IT-DiegoNeural"   #@param ["it-IT-DiegoNeural", "it-IT-GiuseppeNeural"]
EN_RATE       = "-5%"    #@param {type:"string"}
IT_RATE       = "-10%"   #@param {type:"string"}
SLOW_RATE     = "-35%"   #@param {type:"string"}

#@markdown **Pauses** (ms).  `d_it` = duration of the Italian clip.
PAUSE_SCALE    = 1.0    #@param {type:"slider", min:0.6, max:1.8, step:0.05}
THINK_MS       = 1500   #@param {type:"integer"}
SAY_FACTOR     = 1.5    #@param {type:"number"}
MIN_ANSWER_MS  = 2500   #@param {type:"integer"}
MAX_ANSWER_MS  = 12000  #@param {type:"integer"}
REPEAT_BASE_MS = 1000   #@param {type:"integer"}
REPEAT_FACTOR  = 1.3    #@param {type:"number"}
MIN_REPEAT_MS  = 1800   #@param {type:"integer"}
MAX_REPEAT_MS  = 10000  #@param {type:"integer"}
USE_PIANO      = True   #@param {type:"boolean"}

CFG = dict(ENGINE=ENGINE, VOICE_EN=VOICE_EN, VOICE_IT_F=VOICE_IT_F, VOICE_IT_M=VOICE_IT_M,
           EN_RATE=EN_RATE, IT_RATE=IT_RATE, SLOW_RATE=SLOW_RATE, PAUSE_SCALE=PAUSE_SCALE,
           THINK_MS=THINK_MS, SAY_FACTOR=SAY_FACTOR, MIN_ANSWER_MS=MIN_ANSWER_MS, MAX_ANSWER_MS=MAX_ANSWER_MS,
           REPEAT_BASE_MS=REPEAT_BASE_MS, REPEAT_FACTOR=REPEAT_FACTOR,
           MIN_REPEAT_MS=MIN_REPEAT_MS, MAX_REPEAT_MS=MAX_REPEAT_MS,
           NARRATION_GAP_MS=700, DIALOGUE_GAP_MS=600, SECTION_GAP_MS=1200,
           # dry-run speech-rate model (calibrated on the edge-tts voices above)
           IT_SYL_PER_S=4.6, EN_WORDS_PER_S=2.7, CLIP_PAD_MS=570,
           CACHE_DIR="/content/tts_cache", USE_PIANO=USE_PIANO,
           PIANO_URL="https://csee.umbc.edu/~simsek/shortpiano.mp3")

## 2 · TTS engine

In [ ]:
import os, re, json, time, hashlib, subprocess, tempfile, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pydub import AudioSegment

# ----------------------------------------------------------------------------
# Script tags (one per line; blank lines and lines starting with # are ignored)
#   (section) Title                 -> marker in the transcript only (no audio)
#   (narrator) English text         -> English narrator. {curly braces} = Italian
#                                      inserted inside the sentence, e.g.
#                                      "The word {ricevimento} means office hours."
#   (it F) testo  / (it M) testo    -> Italian line (dialogue), female / male voice
#   (repeat [x2] [slow] [F|M]) testo-> Italian model + pause for you to REPEAT.
#                                      x2/x3 = say it 2-3 times; slow = the extra
#                                      repetitions are spoken slowly.
#   (prompt [x2] [F|M]) English || Italiano
#                                   -> English cue, pause to ANSWER (length set by
#                                      the Italian), Italian answer, pause to REPEAT.
#   (build) zione | iscrizione      -> back-chaining: each piece + repeat pause
#   (pause 3)                       -> silence in seconds
#   (english) / (italian)           -> old tags still work (= narrator / it F)
# ----------------------------------------------------------------------------

VOWELS = "aeiouàèéìíòóùúy"

def italian_syllables(text):
    """Rough Italian syllable count (vowel groups); used for dry-run timing only."""
    words = re.findall(r"[A-Za-zÀ-ÿ']+", text.lower())
    n = 0
    for w in words:
        n += max(1, len(re.findall(f"[{VOWELS}]+", w)))
    return n


class PimsleurTTS:
    def __init__(self, cfg):
        self.cfg = cfg
        os.makedirs(cfg["CACHE_DIR"], exist_ok=True)
        self.piano_clip_path = self._download_piano_clip() if cfg.get("USE_PIANO") else None

    # ------------------------------------------------------------------ parsing
    def parse(self, script):
        events = []          # list of (kind, payload dict)
        voice_toggle = 0
        for raw in script.split("\n"):
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            m = re.match(r"^\(([^)]*)\)\s*(.*)$", line)
            if not m:
                print(f"⚠️ untagged line skipped: {line[:60]}")
                continue
            tag, body = m.group(1).strip().lower(), m.group(2).strip()
            words = tag.split()
            head = words[0]
            reps = 1
            for w in words:
                if re.fullmatch(r"x\d", w):
                    reps = int(w[1])
            slow = "slow" in words
            gender = "M" if "m" in words else ("F" if "f" in words else None)

            if head in ("section",):
                events.append(("section", {"title": body}))
            elif head in ("narrator", "english", "en"):
                events.append(("narrator", {"text": body}))
            elif head in ("it", "italian", "italiano"):
                events.append(("it", {"text": body, "g": gender or "F"}))
            elif head == "repeat":
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("repeat", {"text": body, "g": gender, "reps": reps, "slow": slow}))
            elif head == "prompt":
                if "||" not in body:
                    print(f"⚠️ prompt without '||' skipped: {body[:60]}"); continue
                en, it = [s.strip() for s in body.split("||", 1)]
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("prompt", {"en": en, "it": it, "g": gender, "reps": reps, "slow": slow}))
            elif head == "build":
                pieces = [p.strip().lstrip("-") for p in body.split("|") if p.strip()]
                events.append(("build", {"pieces": pieces, "g": gender or "F"}))
            elif head == "pause":
                sec = float(words[1]) if len(words) > 1 else float(body or 1)
                events.append(("pause", {"ms": int(sec * 1000)}))
            else:
                print(f"⚠️ unknown tag ({tag}) skipped")
        return events

    # ------------------------------------------------------------------ pauses
    def answer_pause(self, it_ms):
        c = self.cfg
        p = c["THINK_MS"] + c["SAY_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_ANSWER_MS"]), c["MAX_ANSWER_MS"]))

    def repeat_pause(self, it_ms):
        c = self.cfg
        p = c["REPEAT_BASE_MS"] + c["REPEAT_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_REPEAT_MS"]), c["MAX_REPEAT_MS"]))

    # ------------------------------------------------------------------ TTS jobs
    def _voice(self, lang, g):
        c = self.cfg
        if lang == "en":
            return c["VOICE_EN"]
        return c["VOICE_IT_M"] if g == "M" else c["VOICE_IT_F"]

    @staticmethod
    def _split_narration(text):
        """'The word {casa} means house' -> [('en','The word'),('it','casa'),('en','means house')]"""
        parts = []
        for i, chunk in enumerate(re.split(r"\{(.*?)\}", text)):
            chunk = chunk.strip()
            if re.search(r"\w", chunk):          # skip punctuation-only pieces
                parts.append(("it" if i % 2 else "en", chunk))
        return parts

    def jobs_for(self, events):
        """All unique (text, lang, gender, slow) clips the lesson needs."""
        jobs = set()
        for kind, d in events:
            if kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    jobs.add((chunk, lang, "F", False))
            elif kind == "it":
                jobs.add((d["text"], "it", d["g"], False))
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                jobs.add((it, "it", d["g"], False))
                if d["slow"]:
                    jobs.add((it, "it", d["g"], True))
                if kind == "prompt":
                    jobs.add((d["en"], "en", "F", False))
            elif kind == "build":
                for p in d["pieces"]:
                    jobs.add((p, "it", d["g"], False))
        return sorted(jobs)

    def _cache_path(self, job):
        text, lang, g, slow = job
        key = json.dumps([text, self._voice(lang, g), slow, self.cfg["ENGINE"],
                          self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"],
                          self.cfg["SLOW_RATE"]])
        return os.path.join(self.cfg["CACHE_DIR"], hashlib.md5(key.encode()).hexdigest() + ".mp3")

    def _synth_edge(self, job, path):
        text, lang, g, slow = job
        rate = self.cfg["SLOW_RATE"] if slow else (self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"])
        cmd = ["edge-tts", f"--voice={self._voice(lang, g)}", f"--rate={rate}",
               f"--text={text}", f"--write-media={path}"]
        proxy = os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy")
        if proxy:                       # only matters behind a proxy; Colab has none
            cmd.insert(1, f"--proxy={proxy}")
        subprocess.run(cmd, check=True, capture_output=True, timeout=60)

    def _synth_gtts(self, job, path):
        from gtts import gTTS
        text, lang, g, slow = job
        gTTS(text=text, lang=lang, slow=slow).save(path)

    @staticmethod
    def _clip_ok(job, path):
        """Reject empty / truncated clips (the online TTS sometimes returns silence)."""
        if not (os.path.exists(path) and os.path.getsize(path) > 0):
            return False
        try:
            a = AudioSegment.from_file(path)
        except Exception:
            return False
        n = italian_syllables(job[0]) if job[1] == "it" else 1.5 * len(job[0].split())
        return len(a) >= 1000 * n / 12 and a.dBFS > -60     # far faster than any voice = broken

    def _synth_one(self, job):
        path = self._cache_path(job)
        if self._clip_ok(job, path):
            return job, path, None
        err = None
        engines = [self.cfg["ENGINE"]] + (["gtts"] if self.cfg["ENGINE"] != "gtts" else [])
        for eng in engines:
            for attempt in range(4):
                try:
                    (self._synth_edge if eng == "edge" else self._synth_gtts)(job, path)
                    if self._clip_ok(job, path):
                        return job, path, None
                    err = f"{eng}: empty or truncated audio"
                except Exception as e:
                    err = f"{eng}: {e}"
                time.sleep(2 * (attempt + 1))      # back off: the free TTS service rate-limits bursts
        return job, None, err

    def synthesize_all(self, events, workers=6):
        jobs = self.jobs_for(events)
        print(f"🗣️  {len(jobs)} unique clips to synthesize (cached ones are reused)...")
        self.clips, self.failed = {}, []
        with ThreadPoolExecutor(workers) as ex:
            for i, (job, path, err) in enumerate(ex.map(self._synth_one, jobs), 1):
                if path:
                    self.clips[job] = AudioSegment.from_file(path)
                else:
                    self.failed.append((job, err))
                    self.clips[job] = AudioSegment.silent(duration=1000)
                if i % 100 == 0 or i == len(jobs):
                    print(f"   {i}/{len(jobs)} done")
        if self.failed:
            print(f"⚠️ {len(self.failed)} clips failed (replaced by 1 s silence), e.g.: {self.failed[0]}")

    def clip(self, text, lang, g="F", slow=False):
        return self.clips[(text, lang, g, slow)]

    # ------------------------------------------------------------------ assembly
    def assemble(self, events):
        c = self.cfg
        S = AudioSegment.silent
        out = S(duration=300)
        transcript = []

        def stamp(label, text):
            ms = len(out)
            transcript.append(f"[{ms//60000:02d}:{(ms//1000)%60:02d}] {label:9s} {text}")

        for kind, d in events:
            if kind == "section":
                stamp("SECTION", "=== " + d["title"] + " ===")
                out += S(duration=c["SECTION_GAP_MS"])
            elif kind == "pause":
                out += S(duration=int(d["ms"] * c["PAUSE_SCALE"]))
            elif kind == "narrator":
                stamp("NARRATOR", d["text"])
                for lang, chunk in self._split_narration(d["text"]):
                    out += self.clip(chunk, lang) + S(duration=120)
                out += S(duration=c["NARRATION_GAP_MS"])
            elif kind == "it":
                stamp(f"IT-{d['g']}", d["text"])
                out += self.clip(d["text"], "it", d["g"]) + S(duration=c["DIALOGUE_GAP_MS"])
            elif kind == "repeat":
                stamp("REPEAT", d["text"] + (f"  (x{d['reps']})" if d["reps"] > 1 else ""))
                normal = self.clip(d["text"], "it", d["g"])
                for r in range(d["reps"]):
                    seg = self.clip(d["text"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "prompt":
                stamp("PROMPT", d["en"] + "  ->  " + d["it"])
                normal = self.clip(d["it"], "it", d["g"])
                out += self.clip(d["en"], "en") + S(duration=self.answer_pause(len(normal)))
                for r in range(d["reps"]):
                    seg = self.clip(d["it"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "build":
                stamp("BUILD", " | ".join(d["pieces"]))
                for p in d["pieces"]:
                    seg = self.clip(p, "it", d["g"])
                    out += seg + S(duration=self.repeat_pause(len(seg)))
        return out, transcript

    # ------------------------------------------------------------------ dry run
    def estimate(self, events):
        """Duration estimate without any TTS (speech-rate model)."""
        c = self.cfg
        pad = c["CLIP_PAD_MS"]          # TTS clips carry some silence at both ends
        it_ms = lambda t, slow=False: pad + 1000 * italian_syllables(t) / (c["IT_SYL_PER_S"] * (0.75 if slow else 1))
        en_ms = lambda t: pad + 1000 * len(t.split()) / c["EN_WORDS_PER_S"]
        total, counts = 0.0, {}
        for kind, d in events:
            counts[kind] = counts.get(kind, 0) + 1
            if kind == "section":
                total += c["SECTION_GAP_MS"]
            elif kind == "pause":
                total += d["ms"] * c["PAUSE_SCALE"]
            elif kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    total += (en_ms(chunk) if lang == "en" else it_ms(chunk)) + 120
                total += c["NARRATION_GAP_MS"]
            elif kind == "it":
                total += it_ms(d["text"]) + c["DIALOGUE_GAP_MS"]
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                if kind == "prompt":
                    total += en_ms(d["en"]) + self.answer_pause(it_ms(it))
                for r in range(d["reps"]):
                    m = it_ms(it, d["slow"] and r > 0)
                    total += m + self.repeat_pause(m)
            elif kind == "build":
                for p in d["pieces"]:
                    m = it_ms(p); total += m + self.repeat_pause(m)
        return total / 1000, counts

    # ------------------------------------------------------------------ main
    def _download_piano_clip(self):
        url = self.cfg.get("PIANO_URL")
        try:
            print("🎹 Downloading piano intro/outro clip...")
            p = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False).name
            urllib.request.urlretrieve(url, p)
            return p
        except Exception as e:
            print(f"⚠️ Could not download piano clip: {e}")
            return None

    def build(self, script, output_file):
        events = self.parse(script)
        self.synthesize_all(events)
        print("🎧 Assembling lesson...")
        audio, transcript = self.assemble(events)
        if self.piano_clip_path and os.path.exists(self.piano_clip_path):
            piano = AudioSegment.from_file(self.piano_clip_path)[:10000]   # first 10 s
            intro = piano.fade_in(1500).fade_out(1500)
            outro = piano.fade_in(1000).fade_out(3000)
            audio = intro + AudioSegment.silent(duration=500) + audio + AudioSegment.silent(duration=800) + outro
        audio.export(output_file, format="mp3", bitrate="96k")
        txt = os.path.splitext(output_file)[0] + "_transcript.txt"
        with open(txt, "w", encoding="utf-8") as f:
            f.write("\n".join(transcript))
        print(f"✅ {output_file}: {len(audio)/60000:.1f} min   |   transcript: {txt}")
        return output_file, txt


## 3 · Lesson script

One line = one audio event. Tags:

| tag | what you hear |
|---|---|
| `(narrator) text {italiano}` | English narrator; text in `{ }` is read by the Italian voice |
| `(it F) …` / `(it M) …` | Italian line (dialogue), female / male |
| `(repeat [x2] [slow] [F/M]) …` | Italian model + repeat pause, 1–3 times; `slow` = extra repetitions spoken slowly |
| `(prompt [x2] [F/M]) English \|\| Italiano` | English cue → answer pause → Italian → repeat pause |
| `(build) zione \| iscrizione` | back-chaining, each piece + repeat pause |
| `(pause 2)` · `(section) Title` | silence · transcript marker |

Voices for `repeat`/`prompt` alternate F/M automatically unless `F` or `M` is given.

In [ ]:
LESSON = r"""
#####################################################################
# EVERYDAY ITALIAN — Lesson 6 (B1)
# Theme: morning routine, personal care, daily activities, hobbies, evenings,
#        politics & government, laws & rules, religion & spirituality,
#        small talk: neighbours, party guests, parents, teacher–parent, teammates
# Grammar: reflexive verbs (present & passato prossimo), causative fare
#          (mi faccio tagliare i capelli), mi piace/mi piacciono, è vietato + inf,
#          passive (è stata approvata), secondo me / credo che + congiuntivo,
#          non credo che, si dice che, prima di + inf, dopo aver + participle
#####################################################################

(section) Opening
(narrator) Everyday Italian. Lesson six.
(narrator) Today you will talk about your day, your hobbies, and your beliefs. You will also learn polite small talk, and how to discuss politics and religion with respect.
(narrator) As always, answer out loud in the pause. Then repeat after the Italian speaker.
(narrator) Listen to this conversation. It is early morning. Luca meets his neighbour, Signora Bianchi, in front of their building.
(pause 1)
(it F) Buongiorno, Luca! Già in piedi a quest'ora?
(it M) Buongiorno, signora. Sì, mi alzo sempre alle sei. Prima di andare al lavoro, vado a correre.
(it F) Beato Lei! Io la mattina non mi sveglio mai prima delle otto.
(it M) Ognuno ha i suoi ritmi. Lei come sta?
(it F) Bene, grazie. Senta, domenica va a votare?
(it M) Sì, certo. Secondo me, votare è un dovere. Ma preferisco non parlare di politica alle sei di mattina!
(it F) Ha ragione! Parliamo d'altro. Stasera c'è la riunione di condominio. Viene?
(it M) Purtroppo no. Stasera ho il colloquio con la maestra di mia figlia.
(it F) Ah, e come va a scuola la piccola?
(it M) Bene. È molto brava in matematica, ma chiacchiera troppo in classe.
(it F) Come il papà! Buona giornata, Luca.
(it M) Grazie, anche a Lei, signora!
(pause 1.5)

(section) Part 1 — The morning routine
(narrator) Let's look at this conversation, piece by piece.
(narrator) Many daily actions use reflexive verbs. To get up is {alzarsi}. To wake up is {svegliarsi}.
(repeat x2 slow) Mi alzo sempre alle sei.
(prompt) Say: I always get up at six. || Mi alzo sempre alle sei.
(prompt) Say: I never wake up before eight. || Non mi sveglio mai prima delle otto.
(narrator) Before doing something is {prima di} plus the infinitive.
(prompt) Say: before going to work, I go running. || Prima di andare al lavoro, vado a correre.
(prompt) The neighbour says: already up at this hour? || Già in piedi a quest'ora?
(narrator) {Beato te}, or formally {beato Lei}, means: lucky you!
(repeat x2) Beato Lei!
(prompt) Say to a friend: lucky you! || Beato te!
(narrator) Now the whole morning routine.
(prompt) Say: I take a shower. Use {farsi la doccia}. || Mi faccio la doccia.
(prompt) Say: I brush my teeth. || Mi lavo i denti.
(prompt) Say: I shave. Use {farsi la barba}. || Mi faccio la barba.
(prompt) Say: I get dressed and I have breakfast. || Mi vesto e faccio colazione.
(prompt) Say: for breakfast I have a cappuccino and a croissant. Use {un cornetto}. || A colazione prendo un cappuccino e un cornetto.
(prompt) Say: I always get up at six. || Mi alzo sempre alle sei.

(narrator) Now in the past. Reflexive verbs use {essere}.
(prompt) Say: this morning I got up late. || Stamattina mi sono alzato tardi.
(prompt) Say: I didn't have time to have breakfast. || Non ho avuto tempo di fare colazione.
(prompt) Say: I got dressed in five minutes. || Mi sono vestito in cinque minuti.
(narrator) For personal care, Italians use {farsi} plus an infinitive. It means someone else does it for you.
(repeat x2 slow) Mi faccio tagliare i capelli.
(prompt) Say: tomorrow I'm getting my hair cut. || Domani mi faccio tagliare i capelli.
(prompt) Say: I go to the barber once a month. || Vado dal barbiere una volta al mese.
(prompt) Say: my wife goes to the hairdresser on Saturday. || Mia moglie va dal parrucchiere il sabato.
(prompt) Say: I go to the gym three times a week. || Vado in palestra tre volte alla settimana.
(prompt) Say: before going to work, I go running. || Prima di andare al lavoro, vado a correre.
(prompt) Say: I never wake up before eight. || Non mi sveglio mai prima delle otto.

(section) Part 2 — Daily activities and evenings
(narrator) Now your day. {Di solito} means: usually.
(prompt) Say: usually I start work at nine. || Di solito comincio a lavorare alle nove.
(prompt) Say: at lunchtime I eat something quickly. || A pranzo mangio qualcosa al volo.
(prompt) Say: I come home at six thirty. || Torno a casa alle sei e mezza.
(narrator) After doing something is {dopo aver} plus the past participle.
(prompt) Say: after having dinner, I take a walk. || Dopo aver cenato, faccio una passeggiata.
(prompt) Say: in the evening we watch a series on TV. || La sera guardiamo una serie in TV.
(prompt) Say: on Friday we go out for an aperitivo. || Il venerdì usciamo per un aperitivo.
(prompt) Say: I go to bed around eleven. || Vado a letto verso le undici.
(prompt) Say: last night I fell asleep on the sofa. Use {addormentarsi}. || Ieri sera mi sono addormentato sul divano.
(prompt) Say: tomorrow I'm getting my hair cut. || Domani mi faccio tagliare i capelli.
(prompt) Say: this morning I got up late. || Stamattina mi sono alzato tardi.

(section) Part 3 — Hobbies
(narrator) Now hobbies. In Italian: {gli hobby}, or {il tempo libero}, free time.
(prompt) Ask a friend: what do you do in your free time? || Che cosa fai nel tempo libero?
(narrator) With one thing, say {mi piace}. With many things, say {mi piacciono}.
(prompt) Say: I like to cook. || Mi piace cucinare.
(prompt) Say: I like crime novels. Use {i gialli}. || Mi piacciono i gialli.
(prompt) Say: I play the guitar. Use {suonare}. || Suono la chitarra.
(prompt) Say: on weekends I go hiking in the mountains. || Nel fine settimana faccio escursioni in montagna.
(narrator) Someone who loves a hobby is {appassionato di}.
(prompt) Say: I am passionate about photography. || Sono appassionato di fotografia.
(prompt) Say: I've been doing gardening for two years. || Faccio giardinaggio da due anni.
(prompt) Say: I would like to learn to dance the tango. || Vorrei imparare a ballare il tango.
(prompt) Say: after having dinner, I take a walk. || Dopo aver cenato, faccio una passeggiata.
(prompt) Say: I go to the gym three times a week. || Vado in palestra tre volte alla settimana.

(section) Part 4 — Politics and government
(narrator) Now politics. Let's start with how Italy is governed. Italy is a parliamentary republic. {Una repubblica parlamentare}.
(narrator) The Parliament has two chambers: {la Camera dei Deputati}, and {il Senato}.
(narrator) The head of state is {il Presidente della Repubblica}. The head of government is {il Presidente del Consiglio}, the prime minister.
(build) siglio | Consiglio | Presidente del Consiglio
(prompt) Say: Italy is a parliamentary republic. || L'Italia è una repubblica parlamentare.
(prompt) Say: the Parliament has two chambers. || Il Parlamento ha due camere.
(narrator) To vote is {votare}. Elections are {le elezioni}. A political party is {un partito}.
(prompt) The neighbour asks, formally: are you going to vote on Sunday? || Domenica va a votare?
(prompt) Say: in my opinion, voting is a duty. || Secondo me, votare è un dovere.
(prompt) Say: the elections will be in the spring. || Le elezioni saranno in primavera.
(prompt) Say: in Italy there are many political parties. || In Italia ci sono molti partiti politici.
(narrator) A law is {una legge}. Parliament approves laws. Notice the passive.
(prompt) Say: the law was approved by Parliament. || La legge è stata approvata dal Parlamento.
(prompt) Say: citizens can also vote in a referendum. || I cittadini possono votare anche in un referendum.
(prompt) Say: I like crime novels. || Mi piacciono i gialli.

(narrator) Now opinions. After {credo che}, use the subjunctive. After {non credo che}, too.
(repeat x2 slow) Credo che sia una buona legge.
(prompt) Say: I think it's a good law. || Credo che sia una buona legge.
(prompt) Say: I don't think it's fair. Use {giusto}. || Non credo che sia giusto.
(prompt) Say: people say that the government will change soon. Use {si dice che}. || Si dice che il governo cambierà presto.
(narrator) Politics can be a delicate topic. Here are some polite phrases.
(prompt) Say: everyone has their own ideas. || Ognuno ha le sue idee.
(prompt) Say: I respect your opinion. Speak informally. || Rispetto la tua opinione.
(prompt) Say: I prefer not to talk about politics at dinner. || Preferisco non parlare di politica a cena.
(prompt) Say: you're right. Let's talk about something else. Formal. || Ha ragione. Parliamo d'altro.
(prompt) Say: in my opinion, voting is a duty. || Secondo me, votare è un dovere.

(section) Part 5 — Laws and rules
(narrator) Rules in daily life. Forbidden is {vietato}. Required is {obbligatorio}.
(prompt) Say: smoking is forbidden here. || Qui è vietato fumare.
(prompt) Read the sign: no parking. Use {divieto}, ban. || Divieto di sosta.
(prompt) Say: wearing a seat belt is required. Use {la cintura}. || La cintura di sicurezza è obbligatoria.
(narrator) In Italy, people sort their rubbish. This is {la raccolta differenziata}.
(build) ziata | differenziata | raccolta differenziata
(prompt) Say: here we sort the rubbish. Use {fare la raccolta differenziata}. || Qui facciamo la raccolta differenziata.
(prompt) Say: the plastic goes out on Tuesday. Use {si butta}. || La plastica si butta il martedì.
(prompt) Say: in our building, you must not make noise after ten. Use {non si può}. || Nel nostro condominio non si può fare rumore dopo le dieci.
(prompt) Say: we have to respect the rules. || Bisogna rispettare le regole.
(prompt) Say: I think it's a good law. || Credo che sia una buona legge.
(prompt) Say: I prefer not to talk about politics at dinner. || Preferisco non parlare di politica a cena.

(section) Part 6 — Religion and spirituality
(narrator) Now religion. Most Italians are Catholic, {cattolici}, but Italy has many religions. A believer is {un credente}.
(prompt) Say: I am a believer, but not practising. || Sono credente, ma non praticante.
(prompt) Say: my grandmother goes to Mass every Sunday. || Mia nonna va a messa ogni domenica.
(prompt) Say: there is a mosque and a synagogue in our city. || Nella nostra città ci sono una moschea e una sinagoga.
(prompt) Say: my neighbour is Buddhist. || Il mio vicino è buddista.
(prompt) Say: I am not religious. || Non sono religioso.
(prompt) Say: it is important to respect all religions. || È importante rispettare tutte le religioni.
(narrator) Spiritual life does not always mean religion. Meditation is {la meditazione}. Inner peace is {la pace interiore}.
(prompt) Say: every morning I meditate for ten minutes. || Ogni mattina medito per dieci minuti.
(prompt) Say: yoga helps me find inner peace. || Lo yoga mi aiuta a trovare la pace interiore.
(prompt) Say: I believe that everything happens for a reason. || Credo che tutto succeda per un motivo.
(prompt) Say: here we sort the rubbish. || Qui facciamo la raccolta differenziata.
(prompt) Say: everyone has their own ideas. || Ognuno ha le sue idee.

(section) Part 7 — Small talk with neighbours
(narrator) Now small talk. With neighbours, Italians often talk about the weather.
(prompt) Say, formally: how hot it is today! || Che caldo oggi!
(prompt) Say: they say it will rain tomorrow. || Dicono che domani pioverà.
(narrator) A residents' meeting is {la riunione di condominio}.
(prompt) The neighbour says, formally: tonight there's the residents' meeting. Are you coming? || Stasera c'è la riunione di condominio. Viene?
(prompt) Answer: unfortunately not. || Purtroppo no.
(prompt) Say, formally: if a package arrives for you, I'll take it. Use {ritirare}. || Se arriva un pacco per Lei, lo ritiro io.
(prompt) Ask, formally: could you water my plants while I'm away? || Potrebbe annaffiare le mie piante mentre sono via?
(prompt) Say, formally: let me know if you need anything. || Mi faccia sapere se ha bisogno di qualcosa.
(prompt) Say: every morning I meditate for ten minutes. || Ogni mattina medito per dieci minuti.

(section) Part 8 — At a party
(narrator) Now a party. Social events are {eventi sociali}. You meet new people, and you make small talk.
(prompt) Ask a guest: how do you know Giulia? Use {conoscere}. || Come conosci Giulia?
(prompt) Answer: we work together. || Lavoriamo insieme.
(prompt) Ask: what do you do for work? || Che lavoro fai?
(prompt) Answer: I am an engineer, and you? || Sono ingegnere, e tu?
(prompt) Say: the food is delicious. Who made it? || Il cibo è buonissimo. Chi l'ha preparato?
(prompt) Say: next week we are invited to a wedding. || La settimana prossima siamo invitati a un matrimonio.
(prompt) Say: I have to go, I'm getting up early tomorrow. || Devo andare, domani mi alzo presto.
(prompt) Say: thank you, it was a lovely evening. || Grazie, è stata una bellissima serata.
(prompt) Say, formally: let me know if you need anything. || Mi faccia sapere se ha bisogno di qualcosa.

(section) Part 9 — Parents and teachers
(narrator) Parents talk at the school gate. The primary school teacher is {la maestra}, or {il maestro}.
(prompt) Ask another parent, informally: what class is your son in? || In che classe è tuo figlio?
(prompt) Answer: he is in third grade. || Fa la terza elementare.
(prompt) Say: the children have too much homework. || I bambini hanno troppi compiti.
(narrator) A parent–teacher meeting is {il colloquio con la maestra}.
(build) loquio | colloquio | il colloquio
(prompt) Say: tonight I have the meeting with my daughter's teacher. || Stasera ho il colloquio con la maestra di mia figlia.
(prompt) Ask the teacher, formally: how is my daughter doing at school? || Come va mia figlia a scuola?
(prompt) The teacher answers: she is very good at maths. || È molto brava in matematica.
(prompt) The teacher adds: but she talks too much in class. || Ma chiacchiera troppo in classe.
(prompt) The teacher says: she needs to work harder in Italian. Use {impegnarsi}. || Deve impegnarsi di più in italiano.
(prompt) Ask, formally: how can we help her at home? || Come possiamo aiutarla a casa?
(prompt) Ask a guest: how do you know Giulia? || Come conosci Giulia?

(section) Part 10 — Teammates
(narrator) Finally, teammates. A five-a-side football game, very popular in Italy, is {il calcetto}. Training is {l'allenamento}.
(prompt) Shout to a teammate: pass me the ball! || Passami la palla!
(prompt) Shout: nice shot! || Bel tiro!
(prompt) Say: we played well, but we lost. || Abbiamo giocato bene, ma abbiamo perso.
(prompt) Say: see you at training on Thursday. || Ci vediamo all'allenamento giovedì.
(prompt) Ask: shall we go for a beer after the match? || Andiamo a bere una birra dopo la partita?
(prompt) Say: I hurt my knee. Use {farsi male}. || Mi sono fatto male al ginocchio.
(prompt) Ask the teacher, formally: how is my daughter doing at school? || Come va mia figlia a scuola?
(prompt) Say: I think it's a good law. || Credo che sia una buona legge.

(section) Part 11 — Review
(narrator) Let's review. First, sentences from earlier lessons.
(prompt) Say: when I was in first year, I always studied in the library. || Quando ero al primo anno, studiavo sempre in biblioteca.
(prompt) Ask: do you have it in another colour? || Ce l'ha in un altro colore?
(prompt) Say: the electricity bill is due on the twentieth. || La bolletta della luce scade il venti.
(prompt) Ask, formally: could you change my room? || Potrebbe cambiarmi la camera?
(prompt) Say: we haven't seen each other for ten years! || Non ci vediamo da dieci anni!
(prompt) Ask, formally: do you mind if I open the window? || Le dispiace se apro il finestrino?
(narrator) Now today's lesson. Answer quickly.
(prompt) Say: I always get up at six. || Mi alzo sempre alle sei.
(prompt) Say: this morning I got up late. || Stamattina mi sono alzato tardi.
(prompt) Say: tomorrow I'm getting my hair cut. || Domani mi faccio tagliare i capelli.
(prompt) Say: after having dinner, I take a walk. || Dopo aver cenato, faccio una passeggiata.
(prompt) Say: I am passionate about photography. || Sono appassionato di fotografia.
(prompt) Say: the law was approved by Parliament. || La legge è stata approvata dal Parlamento.
(prompt) Say: I don't think it's fair. || Non credo che sia giusto.
(prompt) Say: smoking is forbidden here. || Qui è vietato fumare.
(prompt) Say: I am a believer, but not practising. || Sono credente, ma non praticante.
(prompt) Say, formally: if a package arrives for you, I'll take it. || Se arriva un pacco per Lei, lo ritiro io.
(prompt) Say: she needs to work harder in Italian. || Deve impegnarsi di più in italiano.
(prompt) Say: see you at training on Thursday. || Ci vediamo all'allenamento giovedì.

(narrator) Now you are Luca. Answer your neighbour.
(it F) Buongiorno, Luca! Già in piedi a quest'ora?
(prompt M) Say: good morning. Yes, I always get up at six. Before going to work, I go running. || Buongiorno. Sì, mi alzo sempre alle sei. Prima di andare al lavoro, vado a correre.
(it F) Beato Lei! Senta, domenica va a votare?
(prompt M) Say: of course. In my opinion, voting is a duty. || Certo. Secondo me, votare è un dovere.
(it F) Stasera c'è la riunione di condominio. Viene?
(prompt M) Say: unfortunately not. Tonight I have the meeting with my daughter's teacher. || Purtroppo no. Stasera ho il colloquio con la maestra di mia figlia.
(it F) Allora buona giornata!
(prompt M) Say: thank you, you too, signora! || Grazie, anche a Lei, signora!

(narrator) Now listen to the first conversation again.
(pause 1)
(it F) Buongiorno, Luca! Già in piedi a quest'ora?
(it M) Buongiorno, signora. Sì, mi alzo sempre alle sei. Prima di andare al lavoro, vado a correre.
(it F) Beato Lei! Io la mattina non mi sveglio mai prima delle otto.
(it M) Ognuno ha i suoi ritmi. Lei come sta?
(it F) Bene, grazie. Senta, domenica va a votare?
(it M) Sì, certo. Secondo me, votare è un dovere. Ma preferisco non parlare di politica alle sei di mattina!
(it F) Ha ragione! Parliamo d'altro. Stasera c'è la riunione di condominio. Viene?
(it M) Purtroppo no. Stasera ho il colloquio con la maestra di mia figlia.
(it F) Ah, e come va a scuola la piccola?
(it M) Bene. È molto brava in matematica, ma chiacchiera troppo in classe.
(it F) Come il papà! Buona giornata, Luca.
(it M) Grazie, anche a Lei, signora!
(pause 1.5)
(narrator) This is the end of lesson six. Listen to it again tomorrow. {Buona giornata, e a presto!}
"""

## 4 · Dry run (no audio) — check length before synthesizing

In [ ]:
tts = PimsleurTTS(CFG)
events = tts.parse(LESSON)
sec, counts = tts.estimate(events)
print(f"Estimated length: {sec/60:.1f} min   (PAUSE_SCALE={PAUSE_SCALE})")
print(counts)
print(f"Unique clips to synthesize: {len(tts.jobs_for(events))}")

## 5 · Build the MP3 + transcript

In [ ]:
mp3, txt = tts.build(LESSON, f"{LESSON_NAME}.mp3")
from IPython.display import Audio, display
display(Audio(mp3))
try:
    from google.colab import files
    files.download(mp3); files.download(txt)
except ImportError:
    pass

## 6 · Making the next lesson

Copy this notebook, change `LESSON_NAME`, and replace the `LESSON` script. A prompt that works well with an LLM:

> Write a ~45-minute Pimsleur-style Italian lesson at level **B1** on *[topic]*, in this tag format: `(narrator)`, `(it F)`, `(it M)`, `(repeat x2 slow)`, `(prompt) English || Italiano`, `(build) a | b | c`, `(pause n)`, `(section)`. Narration in plain short English sentences; put Italian words inside narration in `{braces}`. Start with a 10–14-line dialogue, break it down with back-chaining for long words, ~120 prompts, re-prompt each new sentence 2–4 times at growing intervals (spaced recall), recycle 10–15 sentences from Lesson *N–1*, include one role-play, and replay the dialogue at the end.

Tips
* Too fast/slow for you? Move `PAUSE_SCALE` (it scales every learner pause) — no need to edit the script.
* Long sentences cap at `MAX_ANSWER_MS`; raise it if you need more time on long prompts.
* `edge-tts` is an unofficial client of the Microsoft Edge read-aloud service; if it breaks, set `ENGINE="gtts"` (single Italian voice, no male/female distinction).
